# Automated Thai GEC Fine-Tuning 🇹🇭🤖

This notebook is optimized for **Google Colab (A100)** to fine-tune Large Language Models on Thai Grammatical Error Correction data.
It uses `xllm` with QLoRA (4-bit) and DeepSpeed.


In [ ]:
# ==========================================
# ⚙️ CONFIGURATION & PARAMETERS
# ==========================================

# 1. Experiment Type
# Options: "FT", "SI", "SI-150%", "SI-200%", "SI-300%"
EXPERIMENT_TYPE = "SI-150%" 

# 2. Model Selection
MODEL_ID = "scb10x/typhoon-7b"

# 3. Training Hyperparameters
NUM_EPOCHS = 3
PER_DEVICE_BATCH_SIZE = 4
GRADIENT_ACCUMULATION = 2
WARMUP_STEPS = 100
SAVE_STEPS = 500

# 4. Credentials & Hub
HUGGINGFACE_TOKEN = "your_hf_token_here" # Get from https://huggingface.co/settings/tokens
HUB_USERNAME = "pakawadeep"

# Push settings
PUSH_TO_HUB_WHILE_TRAINING = True

# Define names for your uploaded models
# By default we generate names based on the model and experiment type, but you can override them here:
LORA_HUB_MODEL_ID = f"{HUB_USERNAME}/{MODEL_ID.split('/')[-1]}-lora-ctfl-th-gec-{EXPERIMENT_TYPE}"
HUB_MODEL_ID = f"{HUB_USERNAME}/{MODEL_ID.split('/')[-1]}-ctfl-th-gec-{EXPERIMENT_TYPE}"

# 5. Data Paths (Google Drive Mounted)
TRAIN_PATH = '/content/drive/MyDrive/NLP Thesis/Data/in-use/train-ctfl-cleaned.csv'
GENERATED_PATH = '/content/drive/MyDrive/NLP Thesis/Data/generated_instruction.csv'


## Installation & Setup

In [ ]:
!pip install torch==2.2.1 torchvision==0.17.1 torchaudio==2.2.1 --index-url https://download.pytorch.org/whl/cu121
!pip install transformers==4.40.2
!pip install peft==0.10.0
!pip install xllm[deepspeed]
!pip install deepspeed==0.13.1


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from huggingface_hub import login
if HUGGINGFACE_TOKEN and HUGGINGFACE_TOKEN != "your_hf_token_here":
    login(token=HUGGINGFACE_TOKEN)
else:
    print("Warning: Hugging Face token not provided or default left unchanged. Models might not download or push correctly.")


## Data Preprocessing

In [ ]:
import pandas as pd

# Load train dataset
df_train = pd.read_csv(TRAIN_PATH)
print(f"Original FT rows: {len(df_train)}")

# Mix generated data based on experiment type
if EXPERIMENT_TYPE != "FT":
    df_gen = pd.read_csv(GENERATED_PATH)
    
    if EXPERIMENT_TYPE == "SI":
        df_gen = df_gen.sample(n=4843, random_state=42)
    elif EXPERIMENT_TYPE == "SI-150%":
        df_gen = df_gen.sample(n=7018, random_state=42)
    elif EXPERIMENT_TYPE == "SI-200%":
        df_gen = df_gen.sample(n=10004, random_state=42)
    elif EXPERIMENT_TYPE == "SI-300%":
        pass # All rows
        
    df_combined = pd.concat([df_train, df_gen], ignore_index=True)
else:
    df_combined = df_train

print(f"Total training rows for {EXPERIMENT_TYPE}: {len(df_combined)}")

# Shuffle the combined dataset
df_combined = df_combined.sample(frac=1, random_state=42).reset_index(drop=True)


In [ ]:
def format_data(df):
    defaultPrompt = '''แก้ไขประโยคต่อไปนี้ให้มีการใช้ภาษาที่ถูกต้อง โดยแก้ไขเฉพาะที่จำเป็น'''
    
    # Check if df has 'text' and 'expected'
    if 'text' not in df.columns or 'expected' not in df.columns:
        raise ValueError("Dataframe must contain 'text' and 'expected' columns.")
        
    df['formatted'] = '### Instruction: ' + defaultPrompt + '\n\n### Input: ' + df['text'].astype(str) + '\n\n### Output: ' + df['expected'].astype(str)
    return df['formatted'].tolist()

formatted_texts = format_data(df_combined)

train_data = []
for sample in formatted_texts:
    train_data.append({"text": sample})

print("Sample:")
print(train_data[0])


## Build & Run Experiment

In [ ]:
import torch
import xllm
from xllm import Config
from xllm.datasets import GeneralDataset
from xllm.experiments import Experiment

cuda_is_available = torch.cuda.is_available()
print(f"X—LLM version: {xllm.__version__}\nTorch version: {torch.__version__}\nCuda is available: {cuda_is_available}")

# Initialize Dataset
train_dataset = GeneralDataset(data=train_data)

config = Config(
    use_gradient_checkpointing=True,
    model_name_or_path=MODEL_ID,
    use_flash_attention_2=False,  
    load_in_4bit=True,
    prepare_model_for_kbit_training=True,
    apply_lora=True,
    warmup_steps=WARMUP_STEPS,
    save_steps=SAVE_STEPS,
    logging_steps=10,
    num_train_epochs=NUM_EPOCHS,

    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    max_length=2048,

    # DeepSpeed stage 2 memory optimization
    deepspeed_stage="stage_2",
    single_gpu=True,

    # Hub config for LoRA adapter
    push_to_hub=PUSH_TO_HUB_WHILE_TRAINING,
    hub_model_id=LORA_HUB_MODEL_ID,
    hub_private_repo=True,
    
    report_to_wandb=False,
)

experiment = Experiment(config=config, train_dataset=train_dataset)


In [ ]:
# Build and Run
experiment.build()
experiment.run()


## Fuse Model and Push to Hub

In [ ]:
# ==========================================
# 🧩 FUSING MODEL & PUSHING FULL WEIGHTS
# ==========================================
import xllm.cli.fuse as fuse_cli

# We create a new config for fusing the LoRA weights into the base model
fusing_config = Config(
    model_name_or_path=MODEL_ID,
    lora_hub_model_id=LORA_HUB_MODEL_ID,
    hub_model_id=HUB_MODEL_ID,
    push_to_hub=True,
    hub_private_repo=True,
)

# Fuse and push to the hub
fuse_cli.cli_run(config=fusing_config)
print(f"Successfully fused and pushed full model to {HUB_MODEL_ID}!")
